# Miniproyecto 6 · Un agente de viajes con herramientas MCP

### Cuando recuperar texto no alcanza

**Maestría en Inteligencia Artificial y Ciencia de Datos · Universidad Icesi · NLP**

Juan José Aguado · Juan David Cruz · Juan Diego Ramírez

## 0. El problema

El Miniproyecto 5 montó un RAG sobre reseñas turísticas y fichas de Wikipedia, y dejó a la vista
un límite que no se arregla con mejores *embeddings*: **hay preguntas que no se responden
recuperando documentos.**

Si alguien pregunta *«¿cuántas reseñas de 1★ tiene Tulum?»* o *«¿qué destino tiene la mejor
valoración media?»*, un RAG recupera cinco reseñas y el modelo **estima** la respuesta a partir
de esa muestra. Eso es inventar con buenos modales. Una agregación no se recupera: **se calcula.**

Aquí entra **MCP** (*Model Context Protocol*), el protocolo que permite exponer herramientas a un
modelo de forma estándar, con servidor, transporte y cliente. Le damos al agente cuatro
herramientas sobre el mismo corpus de las entregas anteriores: dos que **calculan** y dos que
**recuperan**. Con las dos vías disponibles para la misma pregunta, lo que importa no es si la
herramienta funciona, sino otra cosa:

> ### La pregunta
> **¿Cuándo conviene una herramienta y cuándo basta con recuperar texto, y sabe el agente
> elegir?**

### Las cuatro herramientas

| Servidor | Herramienta | Qué hace | Naturaleza |
|---|---|---|---|
| `corpus_mcp_server` | `estadisticas_destino` | conteos, media de estrellas, reparto por tipo | **calcula** |
| | `comparar_destinos` | ranking de destinos por valoración media | **calcula** |
| `busqueda_mcp_server` | `buscar_resenas` | búsqueda semántica con filtros de metadatos | **recupera** |
| | `ficha_destino` | la ficha de Wikipedia del destino | **recupera** |

La pareja `estadisticas_destino` y `buscar_resenas` es deliberada: **son la misma pregunta por dos
caminos distintos**, y ahí se ve si el agente entiende la diferencia entre «cuántos» y «qué
dicen».

### Por qué este caso y no el del guía

El guía de la sesión enseña MCP con una calculadora aritmética y una consulta de clima, y evalúa
el comportamiento del agente leyendo la traza y comentándola. Cambiamos dos cosas.

**Las herramientas operan sobre nuestros datos.** Eso permite calcular la respuesta correcta por
un camino independiente, un `groupby` sobre el `DataFrame`, y compararla con lo que dijo el
agente. Con una calculadora no hay nada que medir: la respuesta la sabe cualquiera.

**La evaluación es automática.** El banco de preguntas declara qué herramienta debería llamarse y
con qué argumentos, y se compara con la traza real. Así se puede medir algo que a ojo no se ve: la
**degradación al añadir herramientas**, que es el problema práctico de todo sistema con
herramientas.

### Hipótesis

> **H1 · la agregación necesita herramienta.** En preguntas de conteo, promedio o ranking, el
> agente con herramientas da la cifra exacta y el RAG puro falla, porque solo ve cinco documentos.
>
> **H2 · el cuello de botella es la elección, no la herramienta.** Una vez llamada, la herramienta
> acierta por construcción. Los errores se concentrarán en no llamarla, llamar la equivocada o
> inventar los argumentos.
>
> **H3 · más herramientas, peor elección.** Al pasar de dos a cuatro, con descripciones que se
> parecen, la tasa de selección correcta baja.

### Mapa del notebook

| Sección | Contenido |
|---|---|
| 1 | Entorno, semilla, Ollama y MCP |
| 2 | El corpus y el EDA mínimo que las herramientas necesitan |
| 3 | Qué es MCP y por qué no es «una función más» |
| 4 | Los dos servidores y sus cuatro herramientas, probadas una por una |
| 5 | Sin herramientas: qué inventa el modelo solo |
| 6 | El agente con herramientas y la lectura de su traza |
| 7 | **Banco de preguntas y evaluación automática** (H1 y H2) |
| 8 | **Dos herramientas frente a cuatro** (H3) |
| 9 | Comparación con el RAG del Miniproyecto 5 |
| 10 | El chatbot en Gradio, con la traza visible |
| 11 | Conclusiones y limitaciones |

> **Continuidad con la serie.** Se reutilizan el corpus `vg055/Rest-Mex2025` y las fichas de
> destino construidas en el Miniproyecto 5, pero **no se hereda ningún bloque de celdas**: aquí no
> hay tarea de clasificación y el protocolo de aquellas entregas no aplica (`docs/DECISIONS.md`,
> D-602).

---

## 1. Entorno y reproducibilidad

<!-- PENDIENTE · bloque B · Debe quedar:
- SEED = 42 y temperatura baja en el agente (con temperatura alta la elección de herramienta
  cambia entre corridas y §7 no se reproduce).
- Detección de GPU y de Ollama, con degradación: sin Ollama, §5 a §10 avisan y se saltan.
- MODELO_LLM = "llama3.2:3b", fijado por etiqueta.
- Rutas relativas a data/ y results/. Nada de rutas locales.
-->

---

## 2. El corpus que ven las herramientas

<!-- PENDIENTE · bloque B · Debe quedar:
- Carga de vg055/Rest-Mex2025 y de data/fichas_destino.json (viene de MP5).
- EDA MÍNIMO, solo lo que las herramientas necesitan: destinos disponibles, reseñas por destino,
  reparto de estrellas. No repetir el EDA de MP1: aquí no se clasifica nada.
- Guardar el corpus en un archivo que los servidores puedan leer al arrancar, para no descargarlo
  dos veces.
-->

---

## 3. Qué es MCP, y por qué no es «una función más»

<!-- PENDIENTE · bloque B · Debe quedar:
- Las cuatro piezas explicadas y VISIBLES: servidor, transporte, cliente, invocación.
- Por qué importa el protocolo y no solo la función: el mismo servidor sirve a cualquier cliente
  compatible, y el modelo no necesita saber nada del código.
- stdio frente a HTTP, y por qué usamos streamable-http (D-603).
Ojo: si en el notebook se importa la función directamente, no es MCP y se cae el criterio 3 de la
rúbrica, que vale 2 puntos.
-->

---

## 4. Los servidores y sus herramientas

<!-- PENDIENTE · bloques A y B · Debe quedar:
- Arranque de los dos servidores desde el notebook (puertos 9090 y 9091) y CELDA DE APAGADO
  idempotente. Un puerto ocupado rompe la siguiente ejecución, y eso golpea los 2 puntos de
  reproducibilidad.
- Cliente MCP explícito que liste las herramientas disponibles.
- Las CUATRO herramientas probadas una por una, con su entrada y su salida a la vista.
Contrato de cada herramienta: mcp_servers/README.md
-->

---

## 5. El modelo solo: qué inventa sin herramientas

<!-- PENDIENTE · bloque B · Debe quedar:
- Las mismas preguntas de agregación hechas al modelo SIN herramientas.
- Comparación con el valor real calculado del DataFrame: aquí se ve la magnitud del invento.
- Es la mitad del experimento de H1, así que las preguntas tienen que ser las mismas que en §7.
-->

---

## 6. El agente con herramientas

<!-- PENDIENTE · bloque B · Debe quedar:
- Agente con langchain-mcp-adapters y create_react_agent de LangGraph.
- Una función que devuelva (respuesta, traza) y que permita LEER qué herramienta se llamó y con
  qué argumentos: de eso dependen §7 y §8 enteros.
- Un ejemplo comentado de cada tipo: agregación, recuperación y hecho.
-->

---

## 7. Banco de preguntas y evaluación automática

<!-- PENDIENTE · bloque C · Debe quedar:
- Banco de preguntas donde cada una declara: herramienta esperada, argumentos esperados y, cuando
  aplica, el valor exacto correcto (calculado del DataFrame, no escrito a mano).
- Métricas: exactitud de la cifra (H1), selección correcta de herramienta, argumentos correctos y
  llamadas innecesarias (H2).
- Dimensionar el banco DESPUÉS de medir cuánto tarda una pregunta: cada una son al menos dos
  pasadas por el modelo. En MP4 una corrida estimada en 20 minutos resultó de dos horas.
-->

---

## 8. Dos herramientas frente a cuatro

<!-- PENDIENTE · bloque C · Debe quedar:
- El mismo banco con solo dos herramientas disponibles y con las cuatro.
- Tabla comparativa de selección correcta, argumentos y llamadas innecesarias.
- Lectura: si la confusión se concentra en la pareja estadisticas_destino / buscar_resenas, el
  culpable son las descripciones, no el modelo. Ver la nota sobre docstrings en
  mcp_servers/README.md.
-->

---

## 9. Comparación con el RAG del Miniproyecto 5

<!-- PENDIENTE · bloque C · Debe quedar:
- Las mismas preguntas pasadas por el RAG de MP5 y por el agente.
- Separar por tipo de pregunta: agregación, experiencia y hecho. La tesis es que cada uno gana en
  un tipo distinto, no que uno sea mejor.
-->

---

## 10. El chatbot

<!-- PENDIENTE · bloque C · Debe quedar:
- Interfaz de Gradio que muestre la respuesta Y la traza: qué herramienta, con qué argumentos.
- Cuando la respuesta venga de documentos, citar destino, tipo y estrellas.
- Guion de preguntas para el video (máximo 2 min), que incluya una llamada a herramienta de
  cálculo y una respuesta con cita de reseñas.
- Celda de cierre de la interfaz.
-->

---

## 11. Conclusiones y limitaciones

<!-- PENDIENTE · bloque C · Debe quedar:
- Respuesta a la pregunta del título.
- Veredicto explícito de H1, H2 y H3: confirmada, parcial o refutada, con la cifra que lo sostiene.
- Qué configuración recomendaríamos y por qué.
- Limitaciones, incluidas las incómodas: el banco es propio, una sola semilla, el agente es un
  modelo de 3B y la selección de herramienta depende de los docstrings que nosotros escribimos.
-->